In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml`
    * `/input/policy/korea-2035/power/coal_ammonia_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_value.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_techs.xml`

# Co-firing

The 11th BPESD projects total hydrogen + ammonia power generation at 15.5 TWh in 2030, 32.8 TWh in 2035, and 43.9 TWh in 2038. However, it does not specify the individual generation amounts for hydrogen and ammonia. In contrast, the 10th BPESD projected hydrogen power generation of 6.1 TWh and ammonia power generation of 6.9 TWh in 2030.

In this study, it is assumed that the annual ratio of hydrogen to ammonia generation in the 11th Basic Plan is the same as the ratio in 2030 under the 10th Basic Plan. Accordingly, the hydrogen generation share is $\frac{6.1}{13} = 46.9\%$, and the ammonia generation share is $53.1\%$.

Also we assume that co-firing technologies are introduced from 2030. In the enhanced ambition scenario, ammonia co-firing is assumed to be phased out. Instead, carbon capture and storage (CCS) is introduced, providing an equivalent contribution to coal-fired generation as that previously attributed to ammonia co-firing.

In [2]:
dictCapTWhH2 = {2020: 0, 2023: 0, 2025: 0, 2030: 15.5*0.469, 2035: 32.8*0.469}

In [3]:
dictCapTWhAmmonia = {2020: 0, 2023: 0, 2025: 0, 2030: 15.5*(1-0.469), 2035: 32.8*(1-0.469)}

In [5]:
years_cap, values_cap = xy(dictCapTWhH2)
years_alt, values_alt = xy(dictCapTWhAmmonia)

fig = go.Figure()

for name, x, y, dash in [
    ("Hydrogen co-firing (LNG)", years_cap, values_cap, None),
    ("Ammonia co-firing (Coal)", years_alt, values_alt, "dash"),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2023, 2030, 2035]
annotations = []
for d in (dictCapTWhH2, dictCapTWhAmmonia):
    for yr in target_years:
        val = d.get(yr)
        if val is not None:
            annotations.append(go.layout.Annotation(
                x=yr, y=val,
                xanchor='center', yanchor='bottom',
                text=f"{val:.1f} TWh",
                showarrow=True, arrowhead=1, ax=0, ay=-20
            ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(title='Year', title_font=dict(size=18), tickfont=dict(size=15)),
    yaxis=dict(title='TWh',  title_font=dict(size=18), tickfont=dict(size=15)),
)

import plotly.io as pio
pio.write_image(fig, "../figure/co-firing.jpg", width=800, height=600, scale=3)
fig.show()

In [18]:
years = [2030, 2035]
values_cab = {y: twh_to_ej_str(dictCapTWhAmmonia[y] * 5) for y in years}
policy_name_cab = "Coal-Ammonia-Blend-Ceiling"
policy_type_cab = "tax"
subsector_name_cab = 'coal'
tech_names_cab = ['coal (conv pul ammonia blend 20%)']

In [19]:
xml_value_cab = build_const_value_xml(
    values_by_year=values_cab,
    policy_name=policy_name_cab,
    policy_type=policy_type_cab,
)

xml_techs_cab = build_const_techs_xml(
    years=[2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name_cab,
    policy_name=policy_name_cab,
    tech_names=tech_names_cab,
    policy_type=policy_type_cab,
)

In [20]:
value_path_cab = f"../../input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml"
techs_path_cab = f"../../input/policy/korea-2035/power/coal_ammonia_blend_const_techs_cp.xml"

write_text(value_path_cab, xml_value_cab)
write_text(techs_path_cab, xml_techs_cab)

print("Wrote:", Path(value_path_cab).expanduser())
print("Wrote:", Path(techs_path_cab).expanduser())

Wrote: ../../input/policy/korea-2035/power/coal_ammonia_blend_const_value_cp.xml
Wrote: ../../input/policy/korea-2035/power/coal_ammonia_blend_const_techs_cp.xml


In [21]:
years = [2030, 2035]
values_ccs = {y: twh_to_ej_str(dictCapTWhAmmonia[y]) for y in years}
policy_name_ccs = "Coal-CCS-Floor"
policy_type_ccs = "subsidy"
subsector_name_ccs = 'coal'
tech_names_ccs = ['coal (conv pul CCS)']

In [22]:
xml_value_ccs = build_const_value_xml(
    values_by_year=values_ccs,
    policy_name=policy_name_ccs,
    policy_type=policy_type_ccs,
)

xml_techs_ccs = build_const_techs_xml(
    years=[2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name_ccs,
    policy_name=policy_name_ccs,
    tech_names=tech_names_ccs,
    policy_type=policy_type_ccs,
)

In [23]:
value_path_ccs = f"../../input/policy/korea-2035/power/coal_CCS_const_value_ep.xml"
techs_path_ccs = f"../../input/policy/korea-2035/power/coal_CCS_const_techs_ep.xml"

write_text(value_path_ccs, xml_value_ccs)
write_text(techs_path_ccs, xml_techs_ccs)

print("Wrote:", Path(value_path_ccs).expanduser())
print("Wrote:", Path(techs_path_ccs).expanduser())

Wrote: ../../input/policy/korea-2035/power/coal_CCS_const_value_ep.xml
Wrote: ../../input/policy/korea-2035/power/coal_CCS_const_techs_ep.xml


In [24]:
years = [2030, 2035]
values_gh2b = {y: twh_to_ej_str(dictCapTWhH2[y] * 2) for y in years}
policy_name_gh2b = "Gas-H2-Blend-Ceiling"
policy_type_gh2b = "tax"
subsector_name_gh2b = 'gas'
tech_names_gh2b = ['gas (CC H2 blend 50%)']

In [25]:
xml_value_gh2b = build_const_value_xml(
    values_by_year=values_gh2b,
    policy_name=policy_name_gh2b,
    policy_type=policy_type_gh2b,
)

xml_techs_gh2b = build_const_techs_xml(
    years=[2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name_gh2b,
    policy_name=policy_name_gh2b,
    tech_names=tech_names_gh2b,
    policy_type=policy_type_gh2b,
)

In [26]:
value_path_gh2b = f"../../input/policy/korea-2035/power/gas_H2_blend_const_value.xml"
techs_path_gh2b = f"../../input/policy/korea-2035/power/gas_H2_blend_const_techs.xml"

write_text(value_path_gh2b, xml_value_gh2b)
write_text(techs_path_gh2b, xml_techs_gh2b)

print("Wrote:", Path(value_path_gh2b).expanduser())
print("Wrote:", Path(techs_path_gh2b).expanduser())

Wrote: ../../input/policy/korea-2035/power/gas_H2_blend_const_value.xml
Wrote: ../../input/policy/korea-2035/power/gas_H2_blend_const_techs.xml
